In [1]:
import os
from ast import literal_eval
import pandas as pd
import spacy
import re
from spacy.tokenizer import Tokenizer
import matplotlib.pyplot as plt
import dill
import numpy as np

In [2]:
data = pd.read_csv('../../data/PFUB_ERLASS_dataset_processed_with_geschaeftsnummer.csv')

In [3]:
data = data[data['is_pfub']==True].reset_index(drop=True)

In [4]:
data

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags
0,Amtsgericht Memmingen\nAbteilung für Zwangsvol...,01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...,approved_attachment_and_transfer_order,amtsgericht memmingen\nabteilung für zwangsvol...,"{'case_slug': '163464253141', 'debtor_name': '...",True,False,"['amtsgericht', 'memmingen', 'abteilung', 'zwa...",amtsgericht memmingen\nabteilung für zwangsvol...
1,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,01.06.2022/1654077659_Doc_01062022_000000001_1...,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '110872558124', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...
2,Amtsgericht Eschweiler\n070071 967072\n-Geschä...,01.06.2022/1654077662_Doc_01062022_000000001_1...,approved_seizure,amtsgericht eschweiler\n070071 967072\n-geschä...,"{'case_slug': '114249164962', 'debtor_name': '...",True,False,"['amtsgericht', 'eschweiler', '-geschäftsstell...",amtsgericht eschweiler\n<NUMBER><PHONE>\n-gesc...
3,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,"['amtsgericht', 'münster', 'amtsgericht', 'mün...",amtsgericht münster\n<NUMBER>- amtsgericht mün...
4,Amtsgericht Viersen\n-Geschäftsstelle-\n-15- A...,01.07.2025/ocr-v2_Allgemeines_Schreiben_R__202...,approved_attachment_and_transfer_order,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,"{'case_slug': '125495503778', 'debtor_name': '...",True,False,"['amtsgericht', 'viersen', '-geschäftsstelle-'...",amtsgericht viersen\n-geschäftsstelle-\n<NUMBE...
...,...,...,...,...,...,...,...,...,...
463,Amtsgericht Wuppertal\n-44- Amtsgericht Wupper...,31.01.2023/1675166710_YA_003_31012023_124814.pdf,approved_seizure,amtsgericht wuppertal\n-44- amtsgericht wupper...,"{'case_slug': '110454161073', 'debtor_name': '...",True,False,"['amtsgericht', 'wuppertal', 'amtsgericht', 'w...",amtsgericht wuppertal\n<NUMBER>- amtsgericht w...
464,"Amtsgericht Itzehoe\nAmtsgericht Itzehoe, Berg...",31.01.2023/1675166710_YA_003_31012023_124815.pdf,approved_seizure,"amtsgericht itzehoe\namtsgericht itzehoe, berg...","{'case_slug': None, 'debtor_name': None, 'cred...",True,False,"['amtsgericht', 'itzehoe', 'amtsgericht', 'itz...","amtsgericht itzehoe\namtsgericht itzehoe, berg..."
465,Amtsgericht Leverkusen\n-45- Amtsgericht Lever...,31.05.2023/1685534803_PF_005_31052023_140508.pdf,approved_seizure,amtsgericht leverkusen\n-45- amtsgericht lever...,"{'case_slug': '117578968598', 'debtor_name': '...",True,False,"['amtsgericht', 'leverkusen', 'amtsgerichen', ...",amtsgericht leverkusen\n<NUMBER>- amtsgericht ...
466,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,31.07.2023/1690794316_KD_004_31072023_110320.pdf,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '125677855012', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...


In [5]:
def extract_data(data):
    try:
        data = literal_eval(data)
        case_slug = data['case_slug'] if 'case_slug' in data else None
        debtor_name = data['debtor_name'] if 'debtor_name' in data else None
    except:
        case_slug = None
        debtor_name = None
    return case_slug, debtor_name

data[['case_slug','debtor_name']] = data['data'].apply(lambda x: pd.Series(extract_data(x)))

In [6]:
data.dropna(subset=['debtor_name'], inplace=True)

In [7]:
data

,text,object_key,document_type,cleaned_text,data,is_pfub,is_ladung,tokenized,text_w_tags,case_slug,debtor_name
0,Amtsgericht Memmingen\nAbteilung für Zwangsvol...,01.04.2025/ocr-v2_54_M_1067_25_Schreiben_vom_3...,approved_attachment_and_transfer_order,amtsgericht memmingen\nabteilung für zwangsvol...,"{'case_slug': '163464253141', 'debtor_name': '...",True,False,"['amtsgericht', 'memmingen', 'abteilung', 'zwa...",amtsgericht memmingen\nabteilung für zwangsvol...,163464253141,Julia Zauner
1,Amtsgericht Frankfurt am Main\nMobiliarzwangsv...,01.06.2022/1654077659_Doc_01062022_000000001_1...,approved_seizure,amtsgericht frankfurt am main\nmobiliarzwangsv...,"{'case_slug': '110872558124', 'debtor_name': '...",True,False,"['amtsgericht', 'frankfurt', 'main', 'mobiliar...",amtsgericht frankfurt am main\nmobiliarzwangsv...,110872558124,"Kaiser, Vanessa"
2,Amtsgericht Eschweiler\n070071 967072\n-Geschä...,01.06.2022/1654077662_Doc_01062022_000000001_1...,approved_seizure,amtsgericht eschweiler\n070071 967072\n-geschä...,"{'case_slug': '114249164962', 'debtor_name': '...",True,False,"['amtsgericht', 'eschweiler', '-geschäftsstell...",amtsgericht eschweiler\n<NUMBER><PHONE>\n-gesc...,114249164962,Pitschel
3,Amtsgericht Münster\n-33- Amtsgericht Münster ...,01.07.2024/ocr-v2_BB_005_01072024_122825.pdf,approved_seizure,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '136916228754', 'debtor_name': '...",True,False,"['amtsgericht', 'münster', 'amtsgericht', 'mün...",amtsgericht münster\n<NUMBER>- amtsgericht mün...,136916228754,Georgina Kaiser
4,Amtsgericht Viersen\n-Geschäftsstelle-\n-15- A...,01.07.2025/ocr-v2_Allgemeines_Schreiben_R__202...,approved_attachment_and_transfer_order,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,"{'case_slug': '125495503778', 'debtor_name': '...",True,False,"['amtsgericht', 'viersen', '-geschäftsstelle-'...",amtsgericht viersen\n-geschäftsstelle-\n<NUMBE...,125495503778,Sultan Aslan
...,...,...,...,...,...,...,...,...,...,...,...
461,Amtsgericht Heinsberg\n4070071967072\n-10- Amt...,30.08.2022/1661853645_Scan_YA_00330082022_1141...,approved_seizure,amtsgericht heinsberg\n4070071967072\n-10- amt...,"{'case_slug': '107672088773', 'debtor_name': '...",True,False,"['amtsgericht', 'heinsberg', 'amtsgericht', 'h...",amtsgericht heinsberg\n<NUMBER>\n<NUMBER>- amt...,107672088773,Seidel
462,Amtsgericht Münster\n-33- Amtsgericht Münster ...,30.10.2025/ocr-v2_c0872fd7aa1c431e.pdf,approved_attachment_and_transfer_order,amtsgericht münster\n-33- amtsgericht münster ...,"{'case_slug': '181060314323', 'debtor_name': '...",True,False,"['amtsgericht', 'münster', 'amtsgericht', 'mün...",amtsgericht münster\n<NUMBER>- amtsgericht mün...,181060314323,Anita Große Leusbrock
463,Amtsgericht Wuppertal\n-44- Amtsgericht Wupper...,31.01.2023/1675166710_YA_003_31012023_124814.pdf,approved_seizure,amtsgericht wuppertal\n-44- amtsgericht wupper...,"{'case_slug': '110454161073', 'debtor_name': '...",True,False,"['amtsgericht', 'wuppertal', 'amtsgericht', 'w...",amtsgericht wuppertal\n<NUMBER>- amtsgericht w...,110454161073,Magner
465,Amtsgericht Leverkusen\n-45- Amtsgericht Lever...,31.05.2023/1685534803_PF_005_31052023_140508.pdf,approved_seizure,amtsgericht leverkusen\n-45- amtsgericht lever...,"{'case_slug': '117578968598', 'debtor_name': '...",True,False,"['amtsgericht', 'leverkusen', 'amtsgerichen', ...",amtsgericht leverkusen\n<NUMBER>- amtsgericht ...,117578968598,Aschenbach


### Directly apply debtor name exraction from ladung

In [8]:
# USE CURRENT DEBTOR NAME EXTRACTOR FOR EXTRACTION
import os


notebook_dir = os.getcwd()
project_dir = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(notebook_dir))))
print("Notebook Directory:", notebook_dir)
print("Project Directory:", project_dir)
os.chdir(project_dir)


Notebook Directory: /Users/melih.gorgulu/Desktop/Projects/intent_recognition/notebooks/after-court/pfub-erlass/param_extraction
Project Directory: /Users/melih.gorgulu/Desktop/Projects/intent_recognition


In [9]:
from src.services.attachment_processing.aftercourt_extractors.ladung.debtor_name_extractor import DebtorNameExtractor
from src.services.attachment_processing.aftercourt_extractors.base import NLPModelManager

debtor_name_extractor = DebtorNameExtractor(NLPModelManager.get_model("de_core_news_md"))

In [10]:
data['extracted_debtor_name'] = data['cleaned_text'].apply(lambda x: debtor_name_extractor.extract(x))

In [11]:
test = data[['cleaned_text','debtor_name','extracted_debtor_name']]

In [12]:
test

,cleaned_text,debtor_name,extracted_debtor_name
0,amtsgericht memmingen\nabteilung für zwangsvol...,Julia Zauner,None
1,amtsgericht frankfurt am main\nmobiliarzwangsv...,"Kaiser, Vanessa",None
2,amtsgericht eschweiler\n070071 967072\n-geschä...,Pitschel,Pitschel Ihrem Antrag Auf Erlass Des Pfändungs...
3,amtsgericht münster\n-33- amtsgericht münster ...,Georgina Kaiser,None
4,amtsgericht viersen\n-geschäftsstelle-\n-15- a...,Sultan Aslan,Aslan Es Wurde Der Pfändungs- U. Überweisungsb...
...,...,...,...
461,amtsgericht heinsberg\n4070071967072\n-10- amt...,Seidel,Seidel
462,amtsgericht münster\n-33- amtsgericht münster ...,Anita Große Leusbrock,None
463,amtsgericht wuppertal\n-44- amtsgericht wupper...,Magner,Magner Erhalten Sie Anbei Die Mit Ihrem Antrag...
465,amtsgericht leverkusen\n-45- amtsgericht lever...,Aschenbach,Aschenbach


In [13]:

import re
from collections import defaultdict
from typing import Optional, List
import spacy
from spacy.matcher import Matcher

from src.services.attachment_processing.aftercourt_extractors.base import BaseExtractor


class DebtorNameExtractorNewPfub(BaseExtractor):
    def __init__(self, nlp_model):
        """
        Initialize the debtor name extractor
        
        Args:
            nlp_model: spaCy language model (e.g., spacy.load("de_core_news_sm"))
        """
        self.nlp = nlp_model
        self.gegen_matcher = Matcher(self.nlp.vocab)
        pattern = [{"LOWER": "gegen"}]
        # Add pattern to match "./." as well
        pattern_dot_slash = [{"LOWER": "./."}]
        self.gegen_matcher.add("DOT_SLASH_PATTERN", [pattern_dot_slash])
        self.gegen_matcher.add("GEGEN_PATTERN", [pattern])
    
    def extract(self, text: str) -> Optional[str]:
        """Extract debtor name from text"""
        if not text:
            return None
        
        # Collect context windows around 'gegen' matches
        context_window = self._collect_context_windows(text)
        # Extract debtor name
        debtor_name = self._extract_validated_debtor_name(context_window) if context_window else None
        
        # Name and Surname is capitalized
        if debtor_name:
            debtor_name = ' '.join(word.capitalize() for word in debtor_name.split())

        return debtor_name
    
    def _is_valid_context(self, left_ctx, right_ctx) -> bool:
        """Check if context contains valid entities (PER/LOC/ORG)"""
        left_entities_labels = [ent.label_ for ent in left_ctx.ents]
        right_entities_labels = [ent.label_ for ent in right_ctx.ents]

        if not (any(label in left_entities_labels for label in ["PER", "LOC", "ORG"]) or 
                any(label in right_entities_labels for label in ["PER", "LOC", "ORG"])):
            return False

        return True

    def _get_context_window(self, doc, start, end, window=10) -> Optional[spacy.tokens.span.Span]:
        """Extract context window around the 'gegen' match"""
        left_start = max(start - window, 0)
        right_end = min(end + window, len(doc))
        left_ctx = doc[left_start:start]
        right_ctx = doc[end:right_end]
        flag = self._is_valid_context(left_ctx, right_ctx)
        return right_ctx

    def _collect_context_windows(self, text: str) -> Optional[List[spacy.tokens.span.Span]]:
        """
        In ladung documents, the classic pattern is <PAIR FINANCE>/ 'gegen' <DEBTOR NAME>.
        Therefore this function collects context windows around 'gegen' matches and returns the first valid one.
        The debtor name is then extracted from this context window.
        Args:
            text (str): Input text to extract context windows from
        Returns:
            Optional[List[spacy.tokens.span.Span]]: List of valid context windows or None if none found
        """
        doc = self.nlp(text)
    
        gegen_matches = self.gegen_matcher(doc)
        if not gegen_matches:
            return None
        context_windows = []
        for _, start, end in gegen_matches:
            context = self._get_context_window(doc, start, end, window=15)
            if context:
                context_windows.append(context)
                
        if not context_windows:
            return None

        return context_windows[0]  # Return only the first valid context window

    def _find_used_delimiter(self, context) -> Optional[str]:
        """Find the most frequently used delimiter in the context"""
        used_delimiter = defaultdict(int)
        for t in context:
            if t.is_punct and not t.is_space:
                used_delimiter[t.text] += 1
        # get the most frequent
        most_frequent = max(used_delimiter, key=used_delimiter.get, default=None)
        return most_frequent

    def _validate_debtor_name(self, debtor_name: str) -> Optional[str]:
        """Clean and validate the extracted debtor name"""
        if not debtor_name:
            return None
            
        debtor_name = debtor_name.lower()
        debtor_name = re.sub(r'\b(herrn?|frau|fräulein)\b', '', debtor_name)
        debtor_name = debtor_name.replace('c/o', '')
        # Remove numbers and punctuation, keep only letters, spaces, and 'ß'
        debtor_name = re.sub(r'[^a-zA-ZÀ-ÿß\s-]', '', debtor_name)
        debtor_name = debtor_name.lstrip()
        # remove after \n detected
        debtor_name = re.sub(r'\n.*', '', debtor_name)
        # Clean up multiple spaces and strip
        debtor_name = re.sub(r'\s+', ' ', debtor_name).strip()
        return debtor_name

    def _contains_salutation(self, text: str) -> bool:
        """Check if text contains German salutations"""
        salutation_patterns = [
            r'\b(herrn?|frau|fräulein)\b',
        ]
        return any(re.search(pattern, text, re.IGNORECASE) for pattern in salutation_patterns)

    def _extract_validated_debtor_name(self, context_window) -> Optional[str]:
        """Extract debtor name from context window"""
        if not context_window:
            return None
        
        debtor_name: str = None
        blocks_by_line_break = context_window.text.split("\n")
        
        if len(blocks_by_line_break)>1:
            first_block = blocks_by_line_break[0].strip()
            second_block = blocks_by_line_break[1].strip()
            if len(first_block)>1:
                if ',' in first_block:
                    # Case: "Mustermann, Max"
                    name, surname = first_block.split(',')[1], first_block.split(',')[0]
                    debtor_name = name.strip() + " " + surname.strip()
                    debtor_name = self._validate_debtor_name(debtor_name)
                else:
                    # No delimiter found, take the first line as debtor name
                    # Special Case: Sometimes we have NAME <Linebreak> Surname
                    # check if we have person entity in second block to cover this cases
                    first_line_break_token_idx = [i for i, token in enumerate(context_window) if token.text == '\n'][0]
                    block_after_linebreak_span = context_window[first_line_break_token_idx:]
                    per_ents_second_block = [token for token in block_after_linebreak_span if token.ent_type_ == "PER" and not token.is_punct and not token.is_space]
                    if per_ents_second_block:
                        per_ent_token_idx = [idx for idx, token in enumerate(context_window) if token == per_ents_second_block[0]][0]
                    else:
                        per_ent_token_idx = -1
                    n_of_tokens_between = per_ent_token_idx - first_line_break_token_idx
                    # If the person entity is within the first 4 tokens after line break, consider it
                    if per_ents_second_block and n_of_tokens_between <=4:
                        debtor_name = first_block + " " + per_ents_second_block[0].text
                        debtor_name = self._validate_debtor_name(debtor_name)
                    else:
                        debtor_name = first_block
                        debtor_name = self._validate_debtor_name(debtor_name)

            elif len(second_block)>1:
                # Case: Context Window starts with line break
                if ',' in second_block:
                    name, surname = second_block.split(',')[0], second_block.split(',')[1]
                    debtor_name = surname.strip() + " " + name.strip()
                    debtor_name = self._validate_debtor_name(debtor_name)
                else:
                    debtor_name = second_block
                    debtor_name = self._validate_debtor_name(debtor_name)
        else:
            per_ent = [ent for ent in context_window.ents if ent.label_ == "PER"]
            if per_ent:
                debtor_name = per_ent[0].text
                debtor_name = self._validate_debtor_name(debtor_name)
        return debtor_name

In [14]:
new_extractor = DebtorNameExtractorNewPfub(NLPModelManager.get_model("de_core_news_md"))

In [16]:
new_extractor.extract(test.iloc[0]['cleaned_text'])

'Julia Zauner'

In [ ]:
# data['extracted_debtor_name_NEW'] = data['cleaned_text'].apply(lambda x: new_extractor.extract(x))

In [ ]:
data['context_window'] = data['cleaned_text'].apply(lambda x: new_extractor._collect_context_windows(x))

## Check context windows to have general idea

In [ ]:
# check context_window = None
context_window_none = data[data['context_window'].isnull()]

In [ ]:
context_window_none

In [ ]:
def print_out_clickable_link(df):
    df = df.reset_index()
    for index, row in df.iterrows():
        print(f"Index: {index}, Link: https://eu-central-1.console.aws.amazon.com/s3/object/pair-scanner?region=eu-central-1&prefix={row['object_key']}")

In [ ]:
print_out_clickable_link(context_window_none)

## Result: CHecked all 25 data, there is no debtor name in them. So theye are fine.

In [ ]:
# check context window
check = data[data['context_window'].notnull()][['context_window','debtor_name']]
check['context_window_text'] = check['context_window'].apply(lambda x: repr(x.text) if x else None)
check['context_window_text'] = check['context_window_text'].apply(lambda x: x.replace('\\n', '<LINE_BREAK>'))
check

In [ ]:
check = data[data['debtor_name']=="Musa Camara"]
check
print_out_clickable_link(check)

In [ ]:
check

In [ ]:
new_extractor.extract(data[data['debtor_name']=="Musa Camara"].iloc[0]['cleaned_text'])

In [ ]:
data['extracted_debtor_name_NEW'] = data['cleaned_text'].apply(lambda x: new_extractor.extract(x))

In [ ]:
check = data[['debtor_name','extracted_debtor_name_NEW','extracted_debtor_name','context_window']]
check